# Mountain NER API Demonstration

This notebook demonstrates the fine-tuned DistilBERT Named Entity Recognition (NER) model, deployed as a FastAPI web service.

**Prerequisites:**
1. Download the `model_artifacts`.
2. Start the local inference container:
   `make build-container`

In [6]:
import time
import requests
import numpy as np
import pandas as pd

In [7]:
API_URL = "http://localhost:8080/predict"

def get_predictions(text):
    start = time.time()
    response = requests.post(API_URL, json={"query": text})
    
    if response.status_code != 200:
        print(f"Error: {response.text}")
        return
        
    data = response.json()
    elapsed_time = (time.time() - start) * 1000 # Convert to milliseconds
    
    # Reconstruct entities from BIO tags for clean display
    entities = []
    current_entity = []
    
    for token_data in data["predictions"]:
        label = token_data["label"]
        token = token_data["token"]
        
        if label == "B-MOUNTAIN":
            if current_entity:
                entities.append(" ".join(current_entity))
            current_entity = [token]
        elif label == "I-MOUNTAIN":
            current_entity.append(token)
        else:
            if current_entity:
                entities.append(" ".join(current_entity))
                current_entity = []
                
    # Catch any trailing entities
    if current_entity:
         entities.append(" ".join(current_entity))
    
    print(f"Original Text: {text}")
    print(f"Extracted Mountains: {entities if entities else 'None found'}")
    print(f"Latency: {elapsed_time:.2f} ms")
    print("-" * 60)
    
    return data["predictions"]

## Standard Inference Tests

In [8]:
# Example 1: Standard geographic fact
get_predictions("The Andes run through several South American countries.")

# Example 2: Multiple descriptive words
get_predictions("I hiked in the Dolomite Mountains, known for their unique rock formations.")

Original Text: The Andes run through several South American countries.
Extracted Mountains: ['Andes']
Latency: 44.03 ms
------------------------------------------------------------
Original Text: I hiked in the Dolomite Mountains, known for their unique rock formations.
Extracted Mountains: ['Dolomite Mountains']
Latency: 31.86 ms
------------------------------------------------------------


[{'token': 'I', 'label': 'O', 'confidence': 0.9991},
 {'token': 'hiked', 'label': 'O', 'confidence': 0.9993},
 {'token': 'in', 'label': 'O', 'confidence': 0.9993},
 {'token': 'the', 'label': 'O', 'confidence': 0.9989},
 {'token': 'Dolomite', 'label': 'B-MOUNTAIN', 'confidence': 0.9911},
 {'token': 'Mountains', 'label': 'I-MOUNTAIN', 'confidence': 0.9532},
 {'token': ',', 'label': 'O', 'confidence': 0.9992},
 {'token': 'known', 'label': 'O', 'confidence': 0.9993},
 {'token': 'for', 'label': 'O', 'confidence': 0.9992},
 {'token': 'their', 'label': 'O', 'confidence': 0.9993},
 {'token': 'unique', 'label': 'O', 'confidence': 0.9993},
 {'token': 'rock', 'label': 'O', 'confidence': 0.9991},
 {'token': 'formations', 'label': 'O', 'confidence': 0.9988},
 {'token': '.', 'label': 'O', 'confidence': 0.9994}]

## Edge Cases

In [9]:
# Edge Case 1: Multiple mountains in one sentence (Testing BIO boundaries)
get_predictions("I climbed Mount Eiger in the Swiss Alps before heading to Mont Blanc.")

# Edge Case 2: Lowercase text (Testing if the model relies entirely on casing)
get_predictions("last summer we drove through the carpathian mountains.")

# Edge Case 3: Tricky Subwords
get_predictions("Is Kilimanjaro the highest peak in Africa?")

# Edge Case 4: False Positive Trap (Capitalized words that aren't mountains)
get_predictions("I went to the Apple Store in New York to buy a new Macbook.")

Original Text: I climbed Mount Eiger in the Swiss Alps before heading to Mont Blanc.
Extracted Mountains: ['Mount Eiger', 'Alps', 'Mont Blanc']
Latency: 25.29 ms
------------------------------------------------------------
Original Text: last summer we drove through the carpathian mountains.
Extracted Mountains: ['carpathian mountains']
Latency: 22.22 ms
------------------------------------------------------------
Original Text: Is Kilimanjaro the highest peak in Africa?
Extracted Mountains: ['Kilimanjaro']
Latency: 24.17 ms
------------------------------------------------------------
Original Text: I went to the Apple Store in New York to buy a new Macbook.
Extracted Mountains: ['Apple Store']
Latency: 24.46 ms
------------------------------------------------------------


[{'token': 'I', 'label': 'O', 'confidence': 0.9991},
 {'token': 'went', 'label': 'O', 'confidence': 0.9992},
 {'token': 'to', 'label': 'O', 'confidence': 0.9991},
 {'token': 'the', 'label': 'O', 'confidence': 0.9987},
 {'token': 'Apple', 'label': 'B-MOUNTAIN', 'confidence': 0.9423},
 {'token': 'Store', 'label': 'I-MOUNTAIN', 'confidence': 0.6569},
 {'token': 'in', 'label': 'O', 'confidence': 0.9992},
 {'token': 'New', 'label': 'O', 'confidence': 0.9989},
 {'token': 'York', 'label': 'O', 'confidence': 0.9985},
 {'token': 'to', 'label': 'O', 'confidence': 0.9991},
 {'token': 'buy', 'label': 'O', 'confidence': 0.9992},
 {'token': 'a', 'label': 'O', 'confidence': 0.9993},
 {'token': 'new', 'label': 'O', 'confidence': 0.9992},
 {'token': 'Macbook', 'label': 'O', 'confidence': 0.9988},
 {'token': '.', 'label': 'O', 'confidence': 0.9993}]

## Latency Analysis

In [10]:
print("--- LATENCY STRESS TEST ---")

latencies = []
test_sentence = "The Rocky Mountains are a major mountain range in western North America."

# Warm up the API
requests.post(API_URL, json={"query": test_sentence})

# Run 50 sequential requests
for _ in range(50):
    start = time.time()
    requests.post(API_URL, json={"query": test_sentence})
    latencies.append((time.time() - start) * 1000)

print(f"Average Inference Latency: {np.mean(latencies):.2f} ms")
print(f"95th Percentile Latency:   {np.percentile(latencies, 95):.2f} ms")
print(f"Max Latency:               {np.max(latencies):.2f} ms")

--- LATENCY STRESS TEST ---
Average Inference Latency: 23.76 ms
95th Percentile Latency:   25.11 ms
Max Latency:               26.36 ms


## Conclusions

### **Project Conclusions & Analysis**

Based on the model training and the inference demonstration above, we can draw several key conclusions:

1. **BIO Tagging Integrity:** The model successfully learned the Entity Boundary problem. As seen in the standard inference examples, the model correctly handles consecutive capitalization and maps `B-` and `I-` tags to isolate distinct geographical entities without bleeding into surrounding words.
2. **Architectural Efficiency (Latency):** The deliberate choice to use `distilbert-base-cased` instead of a larger baseline model paid off in deployment. Running via FastAPI in a CPU-only Docker container, the model achieves a highly viable inference latency, making it suitable for real-time, synchronous production deployment without requiring expensive GPU infrastructure.
3. **Limitation Discovered (False Positive Bias):** During stress testing, the model misclassified "Apple Store" as a mountain. This reveals a learned bias: the model heavily associates capitalized, location-adjacent phrases (e.g., "I went to the [Entity]") with the `MOUNTAIN` class. Because the training dataset was primarily composed of sentences containing mountains or generic background text, it lacks sufficient "hard negative" examples.

---

#### **Future Improvements**

To resolve the false positive bias and improve production precision, the following steps should be implemented in the next training iteration:

* **Data Augmentation with "Hard Negatives":** The synthetic dataset generation should be updated to deliberately include sentences with diverse Named Entities (Organizations, People, standard Cities/Countries, and Brands) explicitly labeled as `O` (Outside). This forces the model to learn the semantic difference between a commercial entity and a geographical mountain, rather than relying too heavily on capitalization.
* **Transfer Learning from a General NER Baseline:** Instead of fine-tuning from the raw `distilbert-base-cased` model, training could be initialized from a model already fine-tuned on the CoNLL-03 dataset (e.g., `dslim/bert-base-NER`). This would provide the network with pre-existing, foundational knowledge of generic Organizations and Locations, allowing it to easily filter out entities like an "Apple Store" before applying the specialized mountain classification.